<a href="https://colab.research.google.com/github/habibiputrar/BigData26_A_2411531001_Habibi-Putra-Rizqullah-/blob/main/Praktikum05/BD_A_P05_2411531001_Habibi_Putra_Rizqullah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/habibiputrar/BigData26_A_2411531001_Habibi-Putra-Rizqullah-/blob/main/Praktikum05/BD_A_P05_2411531001_Habibi_Putra_Rizqullah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRAKTIKUM BIG DATA 5
## Pipeline Data Berita: dari Ingestion hingga MapReduce

**Nama:** Habibi Putra Rizqullah  
**NIM:** 2411531001  
**Kelas:** A

In [31]:
from pathlib import Path
import shutil
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DIR_SIMPAN = Path('/content/drive/MyDrive/BigData/Praktikum5')
except ImportError:
    DIR_SIMPAN = Path.cwd() / 'hasil_praktikum5'
DIR_SIMPAN.mkdir(parents=True, exist_ok=True)
print('Folder hasil:', DIR_SIMPAN)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder hasil: /content/drive/MyDrive/BigData/Praktikum5
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder hasil: /content/drive/MyDrive/BigData/Praktikum5


## H-1 Alur pengolahan data

Data berita diambil, dibersihkan, disimpan, lalu diolah untuk mencari kata yang sering muncul. Hasil akhirnya disimpan ke database SQLite.

In [32]:
%pip install -q pandas pyarrow duckdb requests beautifulsoup4 lxml


In [33]:
import os, re, json, time, zlib, queue, threading, collections, shutil, sqlite3, hashlib
from pathlib import Path
from urllib.parse import urlparse, parse_qs
from urllib.robotparser import RobotFileParser
import http.server, html
import requests
import pandas as pd
import pyarrow.dataset as ds
import duckdb
from IPython.display import display, Markdown

RAW = Path('lapisan_mentah')
TERKURASI = Path('lapisan_terkurasi')
SIAP = Path('siap_pakai')
SIAP.mkdir(exist_ok=True)
UA = 'KelasBigData-UNAND/1.0 (praktikum; kontak: hpr.engineer@gmail.com)'

## H-2 Data dan portal lokal

Data awal berupa file CSV dengan kolom tanggal, URL, judul, dan kategori. Data ini ditampilkan melalui portal lokal dengan isi 100 judul per halaman.

In [34]:
URL_DATA = 'https://raw.githubusercontent.com/ibamibrahim/dataset-judul-berita-indonesia/master/detik_news_title.csv'
csv_path = Path('detik_news_title.csv')
if not csv_path.exists():
    r = requests.get(URL_DATA, timeout=90); r.raise_for_status()
    csv_path.write_bytes(r.content)
df_sumber = pd.read_csv(csv_path)
assert {'date','url','title','category'} <= set(df_sumber.columns)
print('Ukuran berkas :', round(csv_path.stat().st_size / 1e6, 1), 'MB')
print('Baris, kolom :', df_sumber.shape)
print('Kolom        :', list(df_sumber.columns))
HALAMAN_UKURAN = 100

def jalankan_portal(df, port=8000):
    records = df.to_dict('records')
    n_hal = (len(records) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN
    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args): pass
        def kirim(self, status, teks, tipe='text/html; charset=utf-8'):
            data = teks.encode('utf-8')
            self.send_response(status); self.send_header('Content-Type', tipe)
            self.send_header('Content-Length', str(len(data))); self.end_headers()
            self.wfile.write(data)
        def do_GET(self):
            u = urlparse(self.path)
            if u.path == '/robots.txt':
                return self.kirim(200, 'User-agent: *\nDisallow: /arsip-privat/\n', 'text/plain')
            if u.path == '/arsip-privat/':
                return self.kirim(200, '<p>Halaman privat</p>')
            if u.path != '/berita': return self.kirim(404, 'Tidak ditemukan')
            try: hal = int(parse_qs(u.query).get('hal', ['1'])[0])
            except ValueError: return self.kirim(400, 'Halaman tidak valid')
            if not 1 <= hal <= n_hal: return self.kirim(404, 'Tidak ditemukan')
            isi = []
            for r in records[(hal-1)*HALAMAN_UKURAN:hal*HALAMAN_UKURAN]:
                iso = pd.to_datetime(r['date'], format='%m/%d/%Y').strftime('%Y-%m-%d')
                isi.append('<article class="item">'
                    f'<h2 class="judul"><a href="{html.escape(str(r["url"]))}">{html.escape(str(r["title"]))}</a></h2>'
                    f'<span class="kategori">{html.escape(str(r["category"]))}</span>'
                    f'<time datetime="{iso}">{r["date"]}</time></article>')
            self.kirim(200, '<html><body>'+''.join(isi)+f'<nav data-total="{n_hal}"></nav></body></html>')
    srv = http.server.ThreadingHTTPServer(('127.0.0.1', port), Handler)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    return srv, n_hal
if 'server' in globals():
    server.shutdown(); server.server_close()
server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = 'http://127.0.0.1:8000'
print('Portal:', BASE, '| Halaman:', N_HALAMAN)


Ukuran berkas : 16.1 MB
Baris, kolom : (91017, 4)
Kolom        : ['date', 'url', 'title', 'category']
Portal: http://127.0.0.1:8000 | Halaman: 911
Ukuran berkas : 16.1 MB
Baris, kolom : (91017, 4)
Kolom        : ['date', 'url', 'title', 'category']
Portal: http://127.0.0.1:8000 | Halaman: 911


## H-3 Mengambil judul dari portal lokal

Data judul diambil dari halaman HTML. Sebelum mengambil data, robots.txt diperiksa untuk mengetahui halaman yang boleh diakses.

In [35]:
from bs4 import BeautifulSoup

def baca_robots(base):
    r = requests.get(base + '/robots.txt', headers={'User-Agent':UA}, timeout=10)
    r.raise_for_status()
    rp = RobotFileParser(); rp.parse(r.text.splitlines())
    return rp

def parse_halaman(teks_html):
    sup = BeautifulSoup(teks_html, 'html.parser')
    hasil = []
    for a in sup.select('article.item'):
        link = a.select_one('h2.judul a')
        hasil.append({'judul':link.get_text(strip=True), 'url':link['href'],
                      'kategori':a.select_one('span.kategori').get_text(strip=True),
                      'tanggal':a.select_one('time')['datetime']})
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    rp = baca_robots(base)
    with requests.Session() as sesi:
        sesi.headers['User-Agent'] = ua
        for h in halaman:
            url = f'{base}/berita?hal={h}'
            if not rp.can_fetch(ua,url):
                print('Dilarang robots.txt:',url); continue
            for percobaan in range(3):
                try:
                    r = sesi.get(url, timeout=15); r.raise_for_status(); break
                except requests.RequestException:
                    if percobaan == 2: raise
                    time.sleep(0.5*(percobaan+1))
            yield from parse_halaman(r.text)
            time.sleep(delay)
robots = baca_robots(BASE)
for jalur in ['/berita?hal=1','/arsip-privat/']:
    print(f'{jalur:<18} boleh di-crawl? {robots.can_fetch(UA, BASE+jalur)}')
contoh = list(crawl_iter(BASE,range(1,4)))
print('Judul dari 3 halaman:', len(contoh))
for c in contoh[:3]:
    print(c)


/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False
Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}
/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False
Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta',

### Pembahasan H-3

Judul dan tautannya ada di `h2.judul a`. Kategori ada di `span.kategori`, sedangkan tanggal diambil dari atribut `datetime`. Jadi, kode mencari bagian HTML tersebut, lalu menyimpan isinya. File robots.txt dipakai untuk mengecek halaman yang boleh diambil.

## H-4 Mengambil berita dari sitemap

Sitemap digunakan untuk mengambil judul, URL, kategori, tanggal, dan kata kunci. Pengambilan dibatasi 20 judul dengan jeda dua detik antarpermintaan.

In [36]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": urlparse(url).netloc.split(".")[0],
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil

CONTOH_XML = '''<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>
      <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
  </url>
</urlset>'''

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:",
      parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True
{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10

In [37]:
EMAIL_ANDA = 'hpr.engineer@gmail.com'
DELAY = 2.0
BATAS_PER_BAGIAN, BATAS_TOTAL = 7,20
SITEMAP_DIPILIH = ['https://news.detik.com/berita/sitemap_news.xml',
 'https://inet.detik.com/news/sitemap_news.xml','https://sport.detik.com/basket/sitemap_news.xml']
_robots = {}
_request_terakhir = 0.0

def get_sopan(url,sesi):
    global _request_terakhir
    time.sleep(max(0, DELAY-(time.monotonic()-_request_terakhir)))
    try:
        r = sesi.get(url,timeout=25)
    finally: _request_terakhir = time.monotonic()
    if r.status_code in (403,429): raise PermissionError(f'Server menolak: HTTP {r.status_code}')
    r.raise_for_status(); return r

def boleh_diambil(url,sesi):
    host = urlparse(url).netloc
    if host not in _robots:
        r = get_sopan(f'https://{host}/robots.txt',sesi)
        rp = RobotFileParser(); rp.parse(r.text.splitlines()); _robots[host] = rp
    return _robots[host].can_fetch(sesi.headers['User-Agent'],url)

def crawl_detik(sitemaps=None):
    if not EMAIL_ANDA.strip(): raise ValueError('Email belum diisi.')
    hasil, seen = [],set()
    with requests.Session() as sesi:
        sesi.headers['User-Agent'] = f'BigData/1.0 (praktikum pembelajaran; kontak: {EMAIL_ANDA})'
        for sm in (SITEMAP_DIPILIH if sitemaps is None else sitemaps):
            if not boleh_diambil(sm,sesi): raise PermissionError('robots.txt melarang: '+sm)
            entri = parse_sitemap_news(get_sopan(sm,sesi).content)
            n = 0
            for x in entri:
                if not x['judul'] or x['url'] in seen: continue
                seen.add(x['url']); hasil.append(x); n += 1
                if len(hasil) >= BATAS_TOTAL: return hasil
                if n >= BATAS_PER_BAGIAN: break
    return hasil
sampel = []
try:
    sampel = crawl_detik()
    Path('contoh_crawl').mkdir(exist_ok=True)
    Path('contoh_crawl/detik_sitemap.jsonl').write_text(
        ''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in sampel),encoding='utf-8')
    display(pd.DataFrame(sampel))
    print('Jumlah:',len(sampel),'| Kategori:',dict(collections.Counter(x['kategori'] for x in sampel)))
except Exception as e:
    print('Berita terbaru belum berhasil diambil:',type(e).__name__,str(e))
    print('Bagian berikutnya memakai data tahun 2020 dari portal lokal.')


,judul,url,kategori,id,tanggal,kata_kunci
0,Kuasa Hukum Tunjukkan Scan Dokumen Ijazah SD h...,https://news.detik.com/berita/d-8729867/kuasa-...,news,8729867,2026-10-29,"ijazah gibran,ptun jakarta,gibran rakabuming"
1,"Sopir Taksi Online di Tangerang Dibegal, Mobil...",https://news.detik.com/berita/d-8729863/sopir-...,news,8729863,2026-10-29,"tangerang,jabodetabek,taksi online dibegal,pol..."
2,Andre Rosiade: Prabowo Komitmen Benahi Pelayan...,https://news.detik.com/berita/d-8729861/andre-...,news,8729861,2026-10-29,"pelayanan air bersih,kota padang,kualitas air,..."
3,"DKI Diserbu Kendaraan Listrik, Pemprov Siapkan...",https://news.detik.com/berita/d-8729848/dki-di...,news,8729848,2026-10-29,"limbah baterai,kendaraan listrik,pengelolaan l..."
4,"Jalan Baru Batutulis Bogor Diuji Coba, Buka hi...",https://news.detik.com/berita/d-8729835/jalan-...,news,8729835,2026-10-29,"jalan baru batutulis,uji coba jalan,bogor sela..."
5,"Ada Aksi di Sekitar Monas, 9 Rute Transjakarta...",https://news.detik.com/berita/d-8729834/ada-ak...,news,8729834,2026-10-29,"transjakarta,pengalihan rute,layanan transport..."
6,Nenek Asal Brebes Linglung Cari Keluarga di Pa...,https://news.detik.com/berita/d-8729810/nenek-...,news,8729810,2026-10-29,"nenek hilang,panti sosial,polisi bersama masya..."
7,Vladimir Putin Bilang Elon Musk Beneran Jenius,https://inet.detik.com/cyberlife/d-8729743/vla...,inet,8729743,2026-10-29,"elon musk,jenius,vladimir putin,rassvet,starlink"
8,Komdigi Sebut Jawaban Chatbot AI Bukan Gambara...,https://inet.detik.com/law-and-policy/d-872964...,inet,8729648,2026-10-29,"chatbot,media sosial,kecerdasan buatan,algorit..."
9,Samsung Umumkan Pemenang Solve for Tomorrow 20...,https://inet.detik.com/cyberlife/d-8729609/sam...,inet,8729609,2026-10-29,"samsung solve for tomorrow,inovasi teknologi,p..."


Jumlah: 20 | Kategori: {'news': 7, 'inet': 7, 'sport': 6}


,judul,url,kategori,id,tanggal,kata_kunci
0,19 Orang Diamankan Polisi Saat Aksi di Jakarta...,https://news.detik.com/berita/d-8729891/19-ora...,news,8729891,2026-10-29,"polda metro jaya,jabodetabek,jakarta"
1,Massa Aksi Sampaikan Aspirasi di Dekat Patung ...,https://news.detik.com/berita/d-8729887/massa-...,news,8729887,2026-10-29,"unjuk rasa,mahasiswa,universitas trisakti,demo..."
2,Prabowo Beri Bonus Rp 3 M ke Peraih Emas Asian...,https://news.detik.com/berita/d-8729886/prabow...,news,8729886,2026-10-29,"prabowo subianto,bonus atlet,asian games 2026,..."
3,"Paniknya Lansia Saat Kebakaran di Cikini, Semp...",https://news.detik.com/berita/d-8729872/panikn...,news,8729872,2026-10-29,"kebakaran,jabodetabek"
4,Kuasa Hukum Tunjukkan Scan Dokumen Ijazah SD h...,https://news.detik.com/berita/d-8729867/kuasa-...,news,8729867,2026-10-29,"ijazah gibran,ptun jakarta,gibran rakabuming"
5,"Sopir Taksi Online di Tangerang Dibegal, Mobil...",https://news.detik.com/berita/d-8729863/sopir-...,news,8729863,2026-10-29,"tangerang,jabodetabek,taksi online dibegal,pol..."
6,Andre Rosiade: Prabowo Komitmen Benahi Pelayan...,https://news.detik.com/berita/d-8729861/andre-...,news,8729861,2026-10-29,"pelayanan air bersih,kota padang,kualitas air,..."
7,Vladimir Putin Bilang Elon Musk Beneran Jenius,https://inet.detik.com/cyberlife/d-8729743/vla...,inet,8729743,2026-10-29,"elon musk,jenius,vladimir putin,rassvet,starlink"
8,Komdigi Sebut Jawaban Chatbot AI Bukan Gambara...,https://inet.detik.com/law-and-policy/d-872964...,inet,8729648,2026-10-29,"chatbot,media sosial,kecerdasan buatan,algorit..."
9,Samsung Umumkan Pemenang Solve for Tomorrow 20...,https://inet.detik.com/cyberlife/d-8729609/sam...,inet,8729609,2026-10-29,"samsung solve for tomorrow,inovasi teknologi,p..."


Jumlah: 20 | Kategori: {'news': 7, 'inet': 7, 'sport': 6}


### Pembahasan H-4

Berita yang terkumpul berjumlah 20 judul, terdiri dari 7 news, 7 inet, dan 6 sport. Berita news dan inet pada sampel bertanggal 29 Oktober 2026, sedangkan sport memuat berita dengan tanggal lebih lama. Jadi, berita yang diambil bersamaan tidak selalu memiliki tanggal terbit yang sama.

Sitemap sudah menyediakan judul, URL, tanggal, dan kata kunci, sehingga halaman artikel tidak perlu dibuka satu per satu. Isi sampel dapat berubah saat pengambilan berikutnya.

## H-5 Source, channel, dan sink

Source mengambil data dari portal lokal, channel menjadi antrean sementara, dan sink menyimpan data ke JSONL. Satu file berisi paling banyak 5.000 data.

In [38]:
RAW.mkdir(exist_ok=True)
for p in RAW.glob('batch_*.jsonl'): p.unlink()
channel = queue.Queue(maxsize=2000)
STOP = object(); batal = threading.Event(); errors = []
statistik = {'masuk':0,'keluar':0,'file':0,'max_antrean':0}
def put_event(ev):
    while not batal.is_set():
        try: channel.put(ev,timeout=0.2); return
        except queue.Full: pass
    raise RuntimeError('Pipeline dibatalkan')
def source():
    try:
        for ev in crawl_iter(BASE,range(1,N_HALAMAN+1)):
            put_event(ev); statistik['masuk'] += 1
            statistik['max_antrean'] = max(statistik['max_antrean'],channel.qsize())
        put_event(STOP)
    except Exception as e: errors.append(e); batal.set()
def sink(ukuran_batch=5000):
    batch = []
    def tulis():
        statistik['file'] += 1
        p = RAW / f"batch_{statistik['file']:04d}.jsonl"
        p.write_text(''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in batch),encoding='utf-8')
        batch.clear()
    try:
        while not batal.is_set():
            try: ev = channel.get(timeout=0.2)
            except queue.Empty: continue
            if ev is STOP: break
            batch.append(ev); statistik['keluar'] += 1
            if len(batch) >= ukuran_batch: tulis()
        if batch and not batal.is_set(): tulis()
    except Exception as e: errors.append(e); batal.set()
t0 = time.perf_counter()
t_src,t_snk = threading.Thread(target=source),threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
if errors: raise RuntimeError('Pipeline gagal') from errors[0]
assert statistik['masuk'] == statistik['keluar'] == len(df_sumber)
waktu = time.perf_counter() - t0
print('Event masuk channel :', f"{statistik['masuk']:,}")
print('Event ditulis sink  :', f"{statistik['keluar']:,}")
print('File batch          :', statistik['file'], '->', sorted(p.name for p in RAW.glob('*.jsonl'))[:3], '...')
print('Antrean maks <= 2000:', statistik['max_antrean'] <= 2000)
print('Total ukuran mentah :', round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), 'MB')
print('Waktu crawl total   :', round(waktu, 1), 'detik untuk', N_HALAMAN, 'halaman')


Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 57.1 detik untuk 911 halaman
Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 67.5 detik untuk 911 halaman


### Pembahasan H-5

Data yang masuk dan ditulis sama, yaitu 91.017 data. Hasilnya tersimpan dalam 19 file. Jadi, jumlah data sebelum dan sesudah penyimpanan tetap sama.

Waktu pengambilan pada output adalah 49,8 detik. Jumlah data dalam antrean masih di bawah batas 2.000. Pada percobaan ini antrean belum sampai penuh.

## H-6 Memeriksa dan merapikan data

Pemeriksaan mencakup judul kosong, jumlah kata, kategori, tanggal, URL, dan judul duplikat. Setelah dibersihkan, data disimpan sebagai Parquet berdasarkan kategori.

In [39]:
mentah = pd.concat([pd.read_json(p,lines=True) for p in sorted(RAW.glob('batch_*.jsonl'))],ignore_index=True)
KATEGORI_DIKENAL = {'finance','food','health','hot','inet','news','oto','sport','travel'}
def aturan_kualitas(d):
    judul = d['judul'].fillna('')
    return {'judul tidak kosong':judul.str.strip().ne(''),
        'judul 3-30 kata':judul.str.split().str.len().between(3,30),
        'kategori dikenal':d['kategori'].isin(KATEGORI_DIKENAL),
        'tanggal valid':pd.to_datetime(d['tanggal'],format='%Y-%m-%d',errors='coerce').notna(),
        'url artikel valid':d['url'].fillna('').str.match(r'https?://[\w-]+\.detik\.com/.+/d-\d+/'),
        'judul tidak duplikat':~d['judul'].duplicated(keep=False)}
kualitas = pd.DataFrame([{'aturan':k,'gagal':int((~v).sum()),'persen_gagal':float((~v).mean()*100)} for k,v in aturan_kualitas(mentah).items()])
print('Baris mentah:', f'{len(mentah):,}')
for nama, lulus in aturan_kualitas(mentah).items():
    print(f'{nama:<24} gagal: {(~lulus).sum():>6,} ({(~lulus).mean()*100:5.2f}%)')
kualitas.to_csv(SIAP/'kualitas.csv',index=False)
bersih = mentah.copy()
bersih['judul'] = bersih['judul'].str.replace(r'\s+',' ',regex=True).str.strip()
bersih = bersih.drop_duplicates(subset='judul',keep='first')
bersih = bersih[bersih['judul'].str.split().str.len().between(3,30)].copy()
bersih['tanggal'] = pd.to_datetime(bersih['tanggal'],format='%Y-%m-%d')
bersih['bulan'] = bersih['tanggal'].dt.strftime('%Y-%m')
bersih['id'] = bersih['url'].str.extract(r'/d-(\d+)')[0].astype('Int64')
bersih['url'] = bersih['url'].where(bersih['url'].str.contains(r'/d-\d+',regex=True,na=False))
print('Baris setelah dibersihkan:', f'{len(bersih):,}', '| dibuang:', f'{len(mentah)-len(bersih):,}')
def simpan_terpartisi(data,kolom,folder):
    shutil.rmtree(folder,ignore_errors=True)
    data.to_parquet(folder,partition_cols=kolom,index=False)
    files = list(Path(folder).rglob('*.parquet'))
    return len(files),sum(p.stat().st_size for p in files)/1e6
for kolom in [['kategori'],['bulan'],['kategori','bulan']]:
    n, mb = simpan_terpartisi(bersih, kolom, 'uji_partisi')
    print(f'partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB')
shutil.rmtree('uji_partisi')
n, mb = simpan_terpartisi(bersih, ['kategori'], TERKURASI)
print('Disimpan ke', TERKURASI, 'dengan partisi kategori:', n, 'file,', round(mb, 1), 'MB')
dataset = ds.dataset(TERKURASI,format='parquet',partitioning='hive')
print('Pruning finance:',len(list(dataset.get_fragments(filter=ds.field('kategori')=='finance'))),
      'dari',len(list(dataset.get_fragments())))


Baris mentah: 91,017
judul tidak kosong       gagal:      0 ( 0.00%)
judul 3-30 kata          gagal:     78 ( 0.09%)
kategori dikenal         gagal:      0 ( 0.00%)
tanggal valid            gagal:      0 ( 0.00%)
url artikel valid        gagal:  5,640 ( 6.20%)
judul tidak duplikat     gagal:    235 ( 0.26%)
Baris setelah dibersihkan: 90,818 | dibuang: 199
partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB
Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Pruning finance: 1 dari 9
Baris mentah: 91,017
judul tidak kosong       gagal:      0 ( 0.00%)
judul 3-30 kata          gagal:     78 ( 0.09%)
kategori dikenal         gagal:      0 ( 0.00%)
tanggal valid            gagal:      0 ( 0.00%)
url artikel valid        gagal:  5,640 ( 6.20%)
judul tidak duplikat     gagal:    235 ( 0.26%)
Baris setelah dibersihkan: 90,818 | dibuang: 199
partisi ['kategori']         

### Pembahasan H-6

Dari 91.017 baris, ada 78 judul yang jumlah katanya di luar batas 3–30 kata. Sebanyak 5.640 URL tidak valid dan 235 baris termasuk kelompok judul duplikat. Angka 235 bukan jumlah baris yang langsung dibuang, karena pemeriksaan menandai semua salinan judul yang sama, termasuk salinan pertama.

Setelah judul dirapikan, duplikat disisakan satu, dan panjang judul dibatasi, tersisa 90.818 baris. Jadi, ada 199 baris yang dibuang. URL rusak dikosongkan, sementara judulnya tetap dipakai agar kategori inet tidak hilang.

Partisi kategori menghasilkan 9 file berukuran sekitar 8,36 MB. Partisi bulan menghasilkan 6 file, sedangkan kategori dan bulan menghasilkan 54 file dengan ukuran sekitar 8,44 MB. Saat mencari finance, cukup 1 dari 9 file kategori yang dibaca.

## H-7 Membaca data dengan DuckDB

DuckDB digunakan untuk menghitung berita per kategori dan judul yang memuat corona per bulan. Satu baris data juga diubah menjadi bentuk dokumen.

In [40]:
con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet',hive_partitioning=true)")
jumlah_kategori = con.sql('SELECT kategori, COUNT(*) jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC').df()
corona_bulan = con.sql("SELECT bulan,COUNT(*) judul_corona FROM berita WHERE lower(judul) LIKE '%corona%' GROUP BY bulan ORDER BY bulan").df()
print(jumlah_kategori.to_string(index=False))
print()
print(corona_bulan.to_string(index=False))
print()
corona_bulan.to_csv(SIAP/'corona_per_bulan.csv',index=False)
r = bersih.iloc[0]
dokumen = {'id':int(r['id']) if pd.notna(r['id']) else None,'judul':r['judul'],
 'kategori':r['kategori'],'tanggal':str(r['tanggal'].date()),'kata':re.findall(r'[a-z]+',r['judul'].lower())}
print(json.dumps(dokumen,ensure_ascii=False,indent=2))


kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}
kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "

### Pembahasan H-7

Kategori news memiliki 32.321 judul, paling banyak di antara sembilan kategori. Kategori sport paling sedikit, yaitu 2.433 judul. Perbedaan ini membuat ukuran masukan tiap mapper tidak sama.

Judul yang memuat corona naik dari 385 pada Januari menjadi 1.077 pada Februari, lalu mencapai 6.994 pada Maret. Jumlahnya turun menjadi 4.913 pada April dan 2.934 pada Mei. Juni berisi 943 judul, tetapi datanya hanya sampai tanggal 13.

Contoh dokumen memiliki ID 4916114 dengan judul Kemnaker Awasi TKA di Meikarta. Selain informasi berita, dokumen juga menyimpan daftar kata hasil pemisahan judul.

## H-8 MapReduce pada kasus kantin

Catatan pesanan tiga kasir digunakan untuk melihat perubahan data pada tahap map, shuffle, dan reduce.

In [41]:
kasir = [['nasi goreng,1','es teh,2','mie ayam,1','es teh,1'],
 ['nasi goreng,2','es teh,1','es teh,3','mie ayam,1'],
 ['mie ayam,2','nasi goreng,1','es teh,1','nasi goreng,1']]
def map_fn(baris):
    menu,n = baris.split(','); return menu,int(n)
def reduce_fn(k,v): return k,sum(v)
def combiner(pasangan):
    c = collections.Counter()
    for k,n in pasangan: c[k] += n
    return list(c.items())
kantin_map = [[map_fn(b) for b in k] for k in kasir]
kantin_shuffle = collections.defaultdict(list)
for h in kantin_map:
    for k,n in h: kantin_shuffle[k].append(n)
kantin_hasil = dict(reduce_fn(k,v) for k,v in kantin_shuffle.items())
print('Map:',kantin_map); print('Shuffle:',dict(kantin_shuffle)); print('Reduce:',kantin_hasil)
print('Pasangan sebelum/sesudah:',sum(map(len,kantin_map)),sum(len(combiner(h)) for h in kantin_map))
assert kantin_hasil == {'nasi goreng':5,'es teh':8,'mie ayam':4}


Map: [[('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)], [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)], [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]]
Shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}
Reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4}
Pasangan sebelum/sesudah: 12 9
Map: [[('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)], [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)], [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]]
Shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}
Reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4}
Pasangan sebelum/sesudah: 12 9


### Pembahasan H-8

Total pesanan adalah nasi goreng 5 porsi, es teh 8 porsi, dan mie ayam 4 porsi. Jumlah seluruhnya 17 porsi.

Tanpa combiner ada 12 pasangan yang dikirim. Setelah dihitung dulu di tiap kasir, tinggal 9 pasangan. Jumlah pasangan berkurang 25%, tetapi total tiap menu tetap sama.

## H-9 Menghitung kata dalam judul

Setiap file kategori menjadi masukan untuk satu mapper. Judul diubah ke huruf kecil dan dipisah menjadi kata, kemudian jumlah kemunculannya dihitung.

In [42]:
STOPWORD = set('yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn'.split())
def tokenisasi(teks):
    return re.findall(r'[a-z]+', teks.lower())
splits = sorted(Path(TERKURASI).rglob('*.parquet'))
print('Jumlah input split:', len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=['judul']))
    print(f' {p.parent.name:<16} {n:>6,} judul')
def mapper_split(path):
    judul = pd.read_parquet(path, columns=['judul'])['judul']
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]
def mapper_combiner(path):
    c = collections.Counter()
    for t in pd.read_parquet(path, columns=['judul'])['judul']:
        c.update(tokenisasi(t))
    return c
hasil_map = [mapper_split(p) for p in splits]
shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}
print('\nPasangan dari semua mapper:', f'{sum(map(len, hasil_map)):,}')
print('Kata unik (key setelah shuffle):', f'{len(hasil_semua):,}')
print('\n10 kata teratas (semua kata):')
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f' {k:<10} {n:>7,}')
print('\n10 kata teratas (tanpa stopword):')
top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
for k, n in top[:10]:
    print(f' {k:<10} {n:>7,}')
cek = collections.Counter(kata for t in bersih['judul'] for kata in tokenisasi(t))
print('\nSama dengan Counter biasa:', dict(cek) == hasil_semua)
assert dict(cek) == hasil_semua

Jumlah input split: 9
 kategori=finance 14,141 judul
 kategori=food     4,314 judul
 kategori=health   4,914 judul
 kategori=hot     16,290 judul
 kategori=inet     5,577 judul
 kategori=news    32,321 judul
 kategori=oto      4,375 judul
 kategori=sport    2,433 judul
 kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
 di          27,390
 corona      17,239
 ini         10,124
 yang         8,668
 dan          6,785
 ke           5,592
 tak          5,096
 dari         4,697
 jadi         4,486
 untuk        3,723

10 kata teratas (tanpa stopword):
 corona      17,239
 rp           3,154
 virus        3,075
 kasus        2,743
 polisi       2,732
 positif      2,682
 indonesia    2,504
 warga        2,495
 rumah        2,465
 jokowi       2,317

Sama dengan Counter biasa: True
Jumlah input split: 9
 kategori=finance 14,141 judul
 kategori=food     4,314 judul
 kategori=health   4,914 judul
 kategor

### Pembahasan H-9

Sembilan file kategori menghasilkan 825.601 pasangan kata. Setelah kata yang sama dikumpulkan dan dijumlahkan, terdapat 32.275 kata unik.

Kata di paling banyak muncul, yaitu 27.390 kali, disusul corona 17.239 kali. Setelah stopword dibuang dari daftar kata teratas, corona berada di urutan pertama, lalu rp 3.154 kali dan virus 3.075 kali.

Jumlah kemunculan kata berbeda dengan jumlah judul. Satu judul bisa memuat kata yang sama lebih dari sekali.

## H-10 Membandingkan waktu dan combiner

Waktu pengolahan dibandingkan pada satu dan dua proses, dengan serta tanpa combiner. Setiap percobaan diulang tiga kali, lalu diambil waktu tengahnya.

In [43]:
import multiprocessing as mp
import statistics

def jalankan(fn,daftar,proses):
    t0 = time.perf_counter()
    if proses == 1: h = list(map(fn,daftar))
    else:
        if 'fork' not in mp.get_all_start_methods():
            raise RuntimeError('Bagian ini perlu Google Colab atau Linux.')
        with mp.get_context('fork').Pool(proses) as pool: h = pool.map(fn,daftar)
    return h,time.perf_counter()-t0
benchmark = []
for proses in [1,2]:
    for nama,fn in [('tanpa combiner',mapper_split),('dengan combiner',mapper_combiner)]:
        waktu = []
        for _ in range(3):
            h,w = jalankan(fn,splits,proses); waktu.append(w)
        benchmark.append({'proses':proses,'skenario':nama,'median_detik':statistics.median(waktu),'pasangan':sum(map(len,h))})
h2 = [mapper_combiner(p) for p in splits]
total = collections.Counter()
for c in h2: total.update(c)
assert dict(total) == hasil_semua
benchmark_df = pd.DataFrame(benchmark)
print('CPU tersedia:', os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")
for row in benchmark_df.itertuples(index=False):
    nama = f'{row.proses} proses, {row.skenario}'
    print(f'{nama:<28}{row.median_detik:>10.3f}{row.pasangan:>20,}')
print('\nHasil dengan combiner sama persis:', dict(total) == hasil_semua)
benchmark_df.to_csv(SIAP/'benchmark.csv',index=False)


CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner         0.862             825,601
1 proses, dengan combiner        0.655              74,009
2 proses, tanpa combiner         1.469             825,601
2 proses, dengan combiner        0.680              74,009

Hasil dengan combiner sama persis: True
CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner         0.544             825,601
1 proses, dengan combiner        0.581              74,009
2 proses, tanpa combiner         1.469             825,601
2 proses, dengan combiner        1.291              74,009

Hasil dengan combiner sama persis: True


### Pembahasan H-10

Satu proses tanpa combiner membutuhkan 0,542856 detik, sedangkan dua proses membutuhkan 2,022454 detik. Dua proses sekitar 3,73 kali lebih lambat. Ada waktu tambahan untuk membagi pekerjaan dan memindahkan hasil antarproses.

Dengan combiner, waktu satu proses menjadi 0,584507 detik dan dua proses 0,702664 detik. Dua proses masih lebih lambat daripada satu proses. Namun, dibanding dua proses tanpa combiner, waktunya turun sekitar 2,88 kali.

Pasangan yang dikirim turun dari 825.601 menjadi 74.009, berkurang sekitar 91,04%. Manfaat paling jelas ada pada berkurangnya pasangan data. Pada satu proses, combiner justru sedikit menambah waktu.

In [44]:
from IPython.display import Markdown
b = benchmark_df.set_index(['proses','skenario'])
a = b.loc[(1,'tanpa combiner'),'median_detik']
c = b.loc[(2,'tanpa combiner'),'median_detik']
p0 = int(b.loc[(1,'tanpa combiner'),'pasangan'])
p1 = int(b.loc[(1,'dengan combiner'),'pasangan'])
display(Markdown(f"Satu proses tanpa combiner membutuhkan **{a:.4f} detik**, sedangkan dua proses **{c:.4f} detik**. Pada pengukuran ini dua proses {'lebih lambat' if c>a else 'lebih cepat'} dengan rasio waktu {c/a:.2f}. Combiner menurunkan pasangan dari **{p0:,}** menjadi **{p1:,}**, atau pengurangan **{(1-p1/p0)*100:.2f}%**. Hasil hitungannya tetap sama. Waktu ini bisa berbeda di komputer lain. Membagi pekerjaan juga membutuhkan waktu."))

Satu proses tanpa combiner membutuhkan **0.8618 detik**, sedangkan dua proses **1.4690 detik**. Pada pengukuran ini dua proses lebih lambat dengan rasio waktu 1.70. Combiner menurunkan pasangan dari **825,601** menjadi **74,009**, atau pengurangan **91.04%**. Hasil hitungannya tetap sama. Waktu ini bisa berbeda di komputer lain. Membagi pekerjaan juga membutuhkan waktu.

Satu proses tanpa combiner membutuhkan **0.5436 detik**, sedangkan dua proses **1.4692 detik**. Pada pengukuran ini dua proses lebih lambat dengan rasio waktu 2.70. Combiner menurunkan pasangan dari **825,601** menjadi **74,009**, atau pengurangan **91.04%**. Hasil hitungannya tetap sama. Waktu ini bisa berbeda di komputer lain. Membagi pekerjaan juga membutuhkan waktu.

## H-11 Beban reducer

Pasangan kata dibagi ke beberapa reducer. Beban tiap reducer dibandingkan untuk melihat apakah pembagiannya sudah rata.

In [45]:
def partitioner(kata,r=4): return zlib.crc32(kata.encode()) % r
def beban(pasangan_per_mapper,r=4):
    b = [0]*r
    for h in pasangan_per_mapper:
        for k,n in h: b[partitioner(k,r)] += 1
    return b
pasangan_combiner = [list(c.items()) for c in h2]
def tampil(nama, b):
    tot = sum(b)
    print(f'{nama:<22}', ' '.join(f'R{i}={x:>7,} ({x/tot*100:4.1f}%)' for i, x in enumerate(b)))
tampil('tanpa combiner', beban(hasil_map))
tampil('dengan combiner', beban(pasangan_combiner))
print('\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):')
rekap_skew = []
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f' {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%')
    rekap_skew.append({'reducer': r, 'terberat_persen': max(b)/sum(b)*100, 'adil_persen': 100/r})
print('\nBeban satu key (tanpa combiner):')
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f' {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}')


tanpa combiner         R0=240,836 (29.2%) R1=170,066 (20.6%) R2=190,727 (23.1%) R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%) R1= 18,267 (24.7%) R2= 18,530 (25.0%) R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
   4 reducer: terberat 29.17% | bagian adil 25.00%
  16 reducer: terberat  8.82% | bagian adil  6.25%
  64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
 di        27,390 pasangan =  3.3% dari total, di reducer R0
 corona    17,239 pasangan =  2.1% dari total, di reducer R2
 ini       10,124 pasangan =  1.2% dari total, di reducer R0
 yang       8,668 pasangan =  1.0% dari total, di reducer R3
 dan        6,785 pasangan =  0.8% dari total, di reducer R3
tanpa combiner         R0=240,836 (29.2%) R1=170,066 (20.6%) R2=190,727 (23.1%) R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%) R1= 18,267 (24.7%) R2= 18,530 (25.0%) R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian ya

### Pembahasan H-11

Dengan empat reducer, beban terberat tanpa combiner ada pada R0, yaitu 240.836 pasangan atau 29,17%. Beban paling ringan ada pada R1, yaitu 20,60%. Setelah combiner, bagian tiap reducer lebih dekat ke 25%, dengan rentang 24,68% sampai 25,55%.

Saat jumlah reducer menjadi 64, beban terberat tanpa combiner masih 4,28%, sedangkan bagian rata hanya 1,56%. Kata di saja berjumlah 27.390 pasangan atau sekitar 3,32% dari seluruh pasangan. Semua pasangan untuk kata yang sama masuk ke satu reducer, sehingga menambah reducer tidak langsung membagi beban kata tersebut.

Dengan combiner dan 64 reducer, beban terberat turun menjadi 1,86%. Angka ini lebih dekat ke bagian rata, tetapi belum persis sama.

In [46]:
from IPython.display import Markdown

b0 = beban(hasil_map, 4)
b1 = beban(pasangan_combiner, 4)
display(Markdown(
    f'Dengan empat reducer, beban terberat tanpa combiner adalah '
    f'**{100 * max(b0) / sum(b0):.2f}%**, sedangkan dengan combiner '
    f'**{100 * max(b1) / sum(b1):.2f}%**. Bagian rata untuk tiap reducer adalah 25%. '
    'Kata yang sering muncul tetap masuk ke reducer yang sama.'
))

Dengan empat reducer, beban terberat tanpa combiner adalah **29.17%**, sedangkan dengan combiner **25.55%**. Bagian rata untuk tiap reducer adalah 25%. Kata yang sering muncul tetap masuk ke reducer yang sama.

Dengan empat reducer, beban terberat tanpa combiner adalah **29.17%**, sedangkan dengan combiner **25.55%**. Bagian rata untuk tiap reducer adalah 25%. Kata yang sering muncul tetap masuk ke reducer yang sama.

## H-12 Hadoop Streaming

Mapper menghasilkan pasangan kata dan angka. Hasilnya diurutkan dengan sort sebelum masuk ke reducer untuk dijumlahkan.

In [47]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')
print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

!head -n 3 masukan.tsv | python3 mapper.py

!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Baris masukan: 90818
kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1

real	0m3.248s
user	0m3.123s
sys	0m0.101s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785
Hasil pipe sama dengan hasil H-9: True
Baris masukan: 90818
kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1

real	0m2.382s
user	0m2.427s
sys	0m0.106s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785
Hasil pipe sama dengan hasil H-9: True


In [ ]:
%%bash
set -euo pipefail
mkdir -p siap_pakai
exec > >(tee siap_pakai/hadoop_log.txt) 2>&1
for f in masukan.tsv mapper.py reducer.py hasil_pipe.txt; do
  if [ ! -f "$f" ]; then
    echo "File $f belum tersedia. Selesaikan sel H-12 sebelumnya pada runtime yang sama."
    exit 1
  fi
done
if [ ! -x /usr/lib/jvm/java-11-openjdk-amd64/bin/java ]; then
  sudo apt-get update -qq
  sudo apt-get install -y openjdk-11-jdk-headless
fi
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
export HADOOP_HOME="$PWD/hadoop-3.3.6"
if [ ! -x "$HADOOP_HOME/bin/hadoop" ]; then
  wget -q --timeout=60 --tries=2 -O hadoop-3.3.6.tar.gz https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
  tar -xzf hadoop-3.3.6.tar.gz
fi
"$JAVA_HOME/bin/java" -version
"$HADOOP_HOME/bin/hadoop" version
OUT="keluaran_hadoop_$(date +%s%N)"
echo "Folder keluaran: $OUT"
"$HADOOP_HOME/bin/hadoop" jar \
  "$HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar" \
  -D mapreduce.framework.name=local -D fs.defaultFS=file:/// \
  -D mapreduce.job.reduces=1 \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input masukan.tsv -output "$OUT"
ls "$OUT"
cp "$OUT/part-00000" siap_pakai/hasil_hadoop.txt
cp "$OUT/_SUCCESS" siap_pakai/hadoop_SUCCESS
diff <(LC_ALL=C sort hasil_pipe.txt) "$OUT/part-00000" > siap_pakai/hadoop_diff.txt
printf 'SAMA: hasil Hadoop identik dengan hasil pipe.\n' | tee siap_pakai/hadoop_verifikasi.txt
wc -l "$OUT/part-00000"
sort -t$'\t' -k2,2nr "$OUT/part-00000" > siap_pakai/hadoop_urut_jumlah.txt
head -n 5 siap_pakai/hadoop_urut_jumlah.txt

### Pembahasan H-12

Pipe selesai dalam 1,859 detik dan menghasilkan 32.275 baris. Hasil perbandingannya True, jadi hitungan dari mapper, sort, dan reducer sama dengan hasil H-9.

Sort menempatkan kata yang sama secara berurutan. Reducer menambah nilainya sampai kata berganti. Tanpa sort, hitungan satu kata bisa terpisah menjadi beberapa subtotal.

## H-13 Kata teratas tiap kategori

Kata dihitung berdasarkan kategori, lalu disimpan ke Parquet dan SQLite.

In [19]:
def mapper_kategori(path,stopword=None):
    stopword = STOPWORD if stopword is None else stopword
    kat = path.parent.name.split('=',1)[1]
    c = collections.Counter()
    for t in pd.read_parquet(path,columns=['judul'])['judul']:
        c.update((kat,k) for k in tokenisasi(t) if k not in stopword)
    return c

def hitung_kategori(stopword):
    gabung = collections.Counter()
    for p in splits: gabung.update(mapper_kategori(p,stopword))
    return gabung

def top_kategori(gabung):
    rows = []
    for kat in sorted(bersih['kategori'].unique()):
        top = sorted([(n,k) for (c,k),n in gabung.items() if c==kat],reverse=True)[:5]
        rows.append({'kategori':kat,'top5':', '.join(f'{k} ({n:,})' for n,k in top)})
    return pd.DataFrame(rows)
gabung = hitung_kategori(STOPWORD)
for row in top_kategori(gabung).itertuples(index=False):
    print(f'{row.kategori:<8}', row.top5)
hasil_df = pd.DataFrame([(kat,k,n) for (kat,k),n in gabung.items()],columns=['kategori','kata','jumlah'])
hasil_df.to_parquet(SIAP/'kata_per_kategori.parquet',index=False)
with sqlite3.connect(SIAP/'laporan.db') as db:
    hasil_df.to_sql('kata_per_kategori',db,if_exists='replace',index=False)
    print('Baris diekspor:',db.execute('SELECT COUNT(*) FROM kata_per_kategori').fetchone()[0])
    print(db.execute("SELECT kata,jumlah FROM kata_per_kategori WHERE kategori='health' ORDER BY jumlah DESC LIMIT 3").fetchall())


finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)
Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


### Pembahasan H-13

Kata corona paling sering muncul pada kategori news, yaitu 7.539 kali. Pada health jumlahnya 2.590 kali dan finance 2.470 kali. Food memiliki kata teratas makanan sebanyak 699 kali, sedangkan oto didominasi mobil sebanyak 1.205 kali.

Hasil hitungan yang masuk ke SQLite berjumlah 73.601 baris. Pencarian tiga kata teratas pada health menghasilkan corona 2.590, virus 1.040, dan pasien 471. Angkanya sama dengan hasil hitungan sebelum disimpan ke database.

## L. Analisis Hasil

1. Pada H-10, dua proses tanpa combiner membutuhkan 2,022454 detik, sedangkan satu proses 0,542856 detik. Dua proses sekitar 3,73 kali lebih lambat. Membagi pekerjaan dan memindahkan hasil membutuhkan waktu tambahan. Combiner mengurangi pasangan dari 825.601 menjadi 74.009. Pada dua proses, waktunya turun menjadi 0,702664 detik, tetapi masih lebih lambat daripada satu proses dengan combiner, yaitu 0,584507 detik.

2. Dengan 64 reducer, beban terberat tanpa combiner masih 4,28%, padahal bagian rata hanya 1,56%. Kata di sendiri menyumbang sekitar 3,32% dari seluruh pasangan dan selalu menuju satu reducer. Karena itu, menambah reducer tidak otomatis membagi beban kata tersebut. Dengan combiner, beban terberat pada 64 reducer menjadi 1,86%.

3. Kata rp muncul 1.385 kali dan ri 1.125 kali pada finance, sedangkan the muncul 528 kali pada hot. Saya tidak langsung membuang semuanya karena rp masih berguna untuk melihat berita ekonomi. Jika yang dicari adalah topik utama tanpa singkatan atau kata umum bahasa Inggris, ketiganya bisa dicoba sebagai stopword tambahan, lalu hasilnya dibandingkan.

4. Ada 5.640 URL yang tidak valid pada data awal. URL rusak tidak bisa dipakai untuk menghubungkan berita dengan tabel lain karena tidak menunjuk artikel yang jelas. Judul tetap dipertahankan agar kategori inet tidak hilang. Setelah pembersihan, kategori inet masih memiliki 5.577 judul.

5. Partisi kategori menghasilkan 9 file, sedangkan kategori dan bulan menghasilkan 54 file. Jumlah mapper mengikuti file masukan. Terlalu banyak file kecil menambah pekerjaan membuka file dan membagi tugas, meskipun ukuran totalnya hampir sama.

6. Pada percobaan pipe, hasil perbandingan dengan H-9 adalah True. Sort diperlukan karena reducer hanya menjumlahkan kata yang berurutan. Tanpa sort, satu kata bisa menghasilkan beberapa subtotal yang terpisah.

7. Dari sitemap terkumpul 20 judul dengan pembagian 7 news, 7 inet, dan 6 sport. Sitemap sudah berisi judul dan tanggal sehingga tidak perlu membuka semua artikel. Permintaan ke server lebih sedikit dan kode tidak bergantung pada tampilan HTML artikel. Kekurangannya, informasi yang tersedia terbatas pada isi sitemap.

## M. Studi Kasus

Pembahasan corona dalam judul berita meningkat dari 385 judul pada Januari menjadi 1.077 pada Februari, lalu mencapai puncak pada Maret dengan 6.994 judul. Setelah itu jumlahnya turun menjadi 4.913 pada April dan 2.934 pada Mei. Pola ini menunjukkan bahwa corona semakin banyak muncul dalam judul menjelang Maret, kemudian berkurang pada dua bulan berikutnya.

Dari hitungan kata per kategori, news mencatat kemunculan corona paling banyak, yaitu 7.539 kali. Health berada setelahnya dengan 2.590 kali dan finance 2.470 kali. Namun, banyaknya kemunculan kata perlu dibedakan dari banyaknya judul. Pada tabel kategori, terdapat 7.529 judul news yang memuat kata corona. Di health, 2.589 dari 4.914 judul memuat kata tersebut, sekitar 52,69%.

Food terlihat paling berbeda jika dilihat dari lima kata teratasnya. Kata makanan muncul 699 kali dan makan 539 kali, sedangkan corona tidak masuk lima besar. Data ini hanya berisi judul, sehingga belum cukup untuk menyimpulkan seluruh isi pemberitaan. Selain itu, Juni hanya mencakup tanggal 1–13. Jumlah 943 judul pada bulan tersebut tidak bisa langsung dibandingkan dengan bulan penuh.

In [ ]:
mask = bersih['judul'].str.contains(r'\bcorona\b',case=False,regex=True,na=False)
studi_kategori = bersih.assign(memuat_corona=mask).groupby('kategori').agg(
    total_judul=('judul','size'),judul_corona=('memuat_corona','sum')).reset_index()
studi_kategori['persen_corona'] = 100*studi_kategori['judul_corona']/studi_kategori['total_judul']
display(corona_bulan); display(studi_kategori.sort_values('judul_corona',ascending=False))
display(top_kategori(gabung))
studi_kategori.to_csv(SIAP/'studi_kategori.csv',index=False)
print('Puncak berdasarkan query H-7:',corona_bulan.loc[corona_bulan['judul_corona'].idxmax()].to_dict())


## N.1 Menambah stopword

Stopword adalah kata yang tidak ikut dihitung dalam analisis. Percobaan ini menambah lima kata, lalu membandingkan lima kata teratas tiap kategori. Pilihan katanya bisa disesuaikan dengan hal yang ingin dicari.

In [20]:
tambahan = {'rp','ri','the','foto','galeri'}
gabung_baru = hitung_kategori(STOPWORD | tambahan)
perbandingan = top_kategori(gabung).merge(top_kategori(gabung_baru),on='kategori',suffixes=('_awal','_baru'))
print('Tambahan:',sorted(tambahan)); display(perbandingan)
perbandingan.to_csv(SIAP/'latihan_stopword.csv',index=False)


Tambahan: ['foto', 'galeri', 'ri', 'rp', 'the']


,kategori,top5_awal,top5_baru
0,finance,"corona (2,470), rp (1,385), ri (1,125), harga ...","corona (2,470), harga (701), jokowi (642), mau..."
1,food,"makanan (699), makan (539), foto (458), galeri...","makanan (699), makan (539), enak (358), bikin ..."
2,health,"corona (2,590), virus (1,040), pasien (471), i...","corona (2,590), virus (1,040), pasien (471), i..."
3,hot,"corona (1,658), anak (793), rumah (561), menin...","corona (1,658), anak (793), rumah (561), menin..."
4,inet,"corona (845), virus (341), ponsel (309), bikin...","corona (845), virus (341), ponsel (309), bikin..."
5,news,"corona (7,539), polisi (2,244), kasus (2,015),...","corona (7,539), polisi (2,244), kasus (2,015),..."
6,oto,"mobil (1,205), corona (576), motor (432), rp (...","mobil (1,205), corona (576), motor (432), indo..."
7,sport,"corona (228), indonesia (223), motogp (209), m...","corona (228), indonesia (223), motogp (209), m..."
8,travel,"corona (1,023), wisata (525), foto (333), duni...","corona (1,023), wisata (525), dunia (299), tur..."


## N.2 Kata teratas per bulan

Kunci hitungan diubah menjadi bulan dan kata. Hasil Januari dibandingkan dengan Mei untuk melihat kata yang masuk dan keluar dari lima besar.

In [21]:
def mapper_bulan(grup):
    bulan,df = grup; c = collections.Counter()
    for t in df['judul']:
        c.update((bulan,k) for k in tokenisasi(t) if k not in STOPWORD)
    return c
per_bulan = collections.Counter()
for grup in bersih.groupby('bulan'): per_bulan.update(mapper_bulan(grup))
for bulan in sorted(bersih['bulan'].unique()):
    top = sorted([(n,k) for (b,k),n in per_bulan.items() if b==bulan],reverse=True)[:5]
    print(bulan,top)
awal = {k for n,k in sorted([(n,k) for (b,k),n in per_bulan.items() if b=='2020-01'],reverse=True)[:5]}
akhir = {k for n,k in sorted([(n,k) for (b,k),n in per_bulan.items() if b=='2020-05'],reverse=True)[:5]}
print('Keluar top5 Januari → Mei:',sorted(awal-akhir),'| Masuk:',sorted(akhir-awal))


2020-01 [(1220, 'banjir'), (616, 'rp'), (554, 'polisi'), (516, 'jokowi'), (493, 'tahun')]
2020-02 [(1076, 'corona'), (630, 'polisi'), (596, 'virus'), (572, 'rp'), (572, 'banjir')]
2020-03 [(6986, 'corona'), (1340, 'virus'), (925, 'positif'), (749, 'pasien'), (695, 'rumah')]
2020-04 [(4914, 'corona'), (828, 'positif'), (706, 'pandemi'), (689, 'psbb'), (609, 'warga')]
2020-05 [(2935, 'corona'), (640, 'kasus'), (628, 'positif'), (596, 'pandemi'), (565, 'warga')]
2020-06 [(943, 'corona'), (471, 'normal'), (465, 'new'), (315, 'kasus'), (265, 'rp')]
Keluar top5 Januari → Mei: ['banjir', 'jokowi', 'polisi', 'rp', 'tahun'] | Masuk: ['corona', 'kasus', 'pandemi', 'positif', 'warga']


## N.3 Pasangan dua kata

Bigram adalah dua kata yang letaknya bersebelahan, misalnya virus corona. Kata tidak dibuang lebih dulu supaya urutan aslinya tetap terjaga.

In [22]:
def mapper_bigram(path):
    c = collections.Counter(); jumlah = 0
    for t in pd.read_parquet(path,columns=['judul'])['judul']:
        token = tokenisasi(t)
        for a,b in zip(token,token[1:]): c[a+' '+b] += 1; jumlah += 1
    return c,jumlah
bigram = collections.Counter(); sebelum = sesudah = 0
for p in splits:
    c,n = mapper_bigram(p); bigram.update(c); sebelum += n; sesudah += len(c)
print('Sebelum combiner:',sebelum,'| Sesudah:',sesudah)
print('Top10:',bigram.most_common(10))


Sebelum combiner: 734783 | Sesudah: 411217
Top10: [('virus corona', 2915), ('corona di', 2178), ('positif corona', 1804), ('di tengah', 1258), ('pandemi corona', 1075), ('di rumah', 1047), ('new normal', 917), ('di indonesia', 871), ('rp juta', 700), ('hari ini', 627)]


## N.4 Membandingkan jumlah reducer

Percobaan memakai 2, 8, dan 32 reducer. Beban paling besar dibandingkan dengan bagian yang seharusnya diterima kalau pembagiannya rata.

In [23]:
latihan_reducer = []
for r in [2,8,32]:
    for nama,h in [('tanpa combiner',hasil_map),('dengan combiner',pasangan_combiner)]:
        b = beban(h,r)
        latihan_reducer.append({'reducer':r,'skenario':nama,'terberat_persen':100*max(b)/sum(b),
                               'adil_persen':100/r,'rasio_terhadap_adil':max(b)/sum(b)*r})
display(pd.DataFrame(latihan_reducer))
pd.DataFrame(latihan_reducer).to_csv(SIAP/'latihan_reducer.csv',index=False)


,reducer,skenario,terberat_persen,adil_persen,rasio_terhadap_adil
0,2,tanpa combiner,52.272587,50.000,1.045452
1,2,dengan combiner,50.234431,50.000,1.004689
2,8,tanpa combiner,16.833555,12.500,1.346684
3,8,dengan combiner,12.852491,12.500,1.028199
4,32,tanpa combiner,5.822788,3.125,1.863292
5,32,dengan combiner,3.438771,3.125,1.100407


## N.5 Menambah jeda pengambilan data

Sebanyak 50 halaman lokal diberi jeda 0,05 detik. Dari waktunya, dibuat perkiraan untuk 911 halaman dengan jeda satu detik. Angka perkiraan itu belum tentu sama dengan waktu percobaan langsung.

In [24]:
t0 = time.perf_counter()
n50 = sum(1 for _ in crawl_iter(BASE,range(1,min(50,N_HALAMAN)+1),delay=0.05))
durasi50 = time.perf_counter()-t0
hal50 = min(50,N_HALAMAN)
overhead_per_hal = max(0,durasi50/hal50-0.05)
print('Judul:',n50,'| Waktu 50 halaman:',round(durasi50,3),'detik')
print('Jeda saja 911 halaman:',911,'detik =',round(911/60,2),'menit')
print('Estimasi termasuk overhead:',round(N_HALAMAN*(1+overhead_per_hal),2),'detik')


Judul: 5000 | Waktu 50 halaman: 5.295 detik
Jeda saja 911 halaman: 911 detik = 15.18 menit
Estimasi termasuk overhead: 961.92 detik


## N.6 Sitemap lain

Salah satu sumber diganti dengan sitemap dari bagian berita lain. Hasilnya diperiksa menggunakan aturan kualitas yang sama.

In [25]:
JALANKAN_SITEMAP_LAIN = False
SITEMAP_PENGGANTI = ''
sampel_lain = []
if JALANKAN_SITEMAP_LAIN:
    u = urlparse(SITEMAP_PENGGANTI)
    if u.scheme != 'https' or not u.netloc.endswith('.detik.com') or not u.path.endswith('sitemap_news.xml'):
        raise ValueError('Isi URL sitemap_news.xml detik.com yang valid dari indeks sitemap.')
    pilihan = SITEMAP_DIPILIH.copy(); pilihan[-1] = SITEMAP_PENGGANTI
    sampel_lain = crawl_detik(pilihan)
    cek_lain = pd.DataFrame(sampel_lain)
    display(cek_lain)
    if len(cek_lain):
        display(pd.DataFrame([{'aturan':k,'gagal':int((~v).sum())} for k,v in aturan_kualitas(cek_lain).items()]))
        print('Kategori:',cek_lain['kategori'].value_counts().to_dict())
else: print('Sitemap lain belum dicoba. Isi alamatnya lalu ubah pilihannya menjadi True.')


Sitemap lain belum dicoba. Isi alamatnya lalu ubah pilihannya menjadi True.


## N.7 Menghitung kata kunci berita

Kata kunci di sitemap dipisahkan dengan koma. Setiap bagian dihitung sebagai satu kata kunci, termasuk bagian yang berisi beberapa kata.

In [26]:
def mapper_kata_kunci(entri):
    for x in entri:
        for k in x.get('kata_kunci','').split(','):
            k = re.sub(r'\s+',' ',k.strip().lower())
            if k: yield k,1

def reducer_jumlah(pasangan):
    c = collections.Counter()
    for k,n in pasangan: c[k] += n
    return c
if sampel:
    kata_kunci = reducer_jumlah(mapper_kata_kunci(sampel))
    display(pd.DataFrame(kata_kunci.most_common(20),columns=['kata_kunci','jumlah']))
    pd.DataFrame(kata_kunci.items(),columns=['kata_kunci','jumlah']).to_csv(SIAP/'kata_kunci_crawl.csv',index=False)
else: print('Belum ada berita terbaru yang bisa dihitung.')


,kata_kunci,jumlah
0,ibl,4
1,jabodetabek,3
2,aksi di monas,2
3,ibl all indonesian 2026,2
4,ibl all indonesian,2
5,ijazah gibran,1
6,ptun jakarta,1
7,gibran rakabuming,1
8,tangerang,1
9,taksi online dibegal,1


## O. Tugas mandiri: inverted index

Inverted index menyimpan daftar judul yang memuat suatu kata. Contohnya, kata mobil menunjuk ke ID semua judul yang mengandung kata mobil.

Sebagian berita inet tidak punya ID sumber karena tautannya rusak. Karena itu, tiap judul diberi ID dokumen sendiri. ID dibuat dengan SHA-256 dari kategori, tanggal, judul, dan ID sumber jika ada. Data yang sama akan mendapat ID yang sama. ID sumber tetap disimpan di tabel pemetaan.

Dalam satu judul, kata yang sama cukup dicatat sekali. Stopword tetap disimpan supaya indeks bisa dipakai untuk mencari semua kata.

In [27]:
dok = bersih.copy()
def buat_id(r):
    nilai = [str(r['kategori']),str(r['tanggal'].date()),str(r['judul']),
             str(int(r['id'])) if pd.notna(r['id']) else None]
    return 'doc_'+hashlib.sha256(json.dumps(nilai,ensure_ascii=False).encode()).hexdigest()
dok['doc_id'] = dok.apply(buat_id,axis=1)
assert dok['doc_id'].is_unique
pemetaan = dok[['doc_id','id','judul','kategori','tanggal','url']]
pemetaan.to_parquet(SIAP/'pemetaan_dokumen.parquet',index=False)

def mapper_index(df):
    for r in df.itertuples(index=False):
        for kata in set(tokenisasi(r.judul)): yield kata,r.doc_id

def shuffle_index(pasangan):
    kelompok = collections.defaultdict(set)
    for kata,doc_id in pasangan: kelompok[kata].add(doc_id)
    return kelompok

def reducer_index(kata,ids): return kata,sorted(ids)
kelompok_index = collections.defaultdict(set)
for kat,grup in dok.groupby('kategori'):
    lokal = shuffle_index(mapper_index(grup))
    for kata,ids in lokal.items(): kelompok_index[kata].update(ids)
inverted_index = dict(reducer_index(k,ids) for k,ids in kelompok_index.items())
index_df = pd.DataFrame([{'kata':k,'doc_ids':v,'jumlah_dokumen':len(v)} for k,v in sorted(inverted_index.items())])
index_df.to_parquet(SIAP/'inverted_index.parquet',index=False)
contoh_index = [{'kata':k,'jumlah_dokumen':len(inverted_index.get(k,[])),
                 'lima_id_pertama':inverted_index.get(k,[])[:5]} for k in ['corona','virus','mobil']]
display(pd.DataFrame(contoh_index))
(SIAP/'contoh_inverted_index.json').write_text(json.dumps(contoh_index,ensure_ascii=False,indent=2),encoding='utf-8')
assert set(inverted_index) == set(hasil_semua)
assert all(len(ids)==len(set(ids)) for ids in inverted_index.values())
print('Kata unik:',len(inverted_index),'| Dokumen:',len(dok),'| ID sumber kosong:',int(dok['id'].isna().sum()))


,kata,jumlah_dokumen,lima_id_pertama
0,corona,17228,[doc_0001c6a884a9a3d9849d8a9adc2b12c3144fe7fb5...
1,virus,3067,[doc_0002b2604700b97fb0a2566ee9ed41d74f9542e87...
2,mobil,1678,[doc_00411f010f7282957b9568c146f56bb9d08a07e06...


Kata unik: 32275 | Dokumen: 90818 | ID sumber kosong: 5577


## Pemeriksaan hasil

Jumlah baris, file, dan pasangan kata dibandingkan dengan nilai acuan.

In [28]:
pemeriksaan = [
 ('baris sumber',len(df_sumber),91017),('halaman lokal',N_HALAMAN,911),
 ('file batch',statistik['file'],19),('baris bersih',len(bersih),90818),
 ('input split',len(splits),9),('pasangan mapper',sum(map(len,hasil_map)),825601),
 ('kata unik',len(hasil_semua),32275),('pasangan combiner',sum(map(len,h2)),74009),
 ('baris ekspor kategori',len(hasil_df),73601)]
validasi = pd.DataFrame(pemeriksaan,columns=['pemeriksaan','aktual','acuan_modul'])
validasi['sesuai'] = validasi['aktual'].eq(validasi['acuan_modul'])
display(validasi); validasi.to_csv(SIAP/'validasi.csv',index=False)


,pemeriksaan,aktual,acuan_modul,sesuai
0,baris sumber,91017,91017,True
1,halaman lokal,911,911,True
2,file batch,19,19,True
3,baris bersih,90818,90818,True
4,input split,9,9,True
5,pasangan mapper,825601,825601,True
6,kata unik,32275,32275,True
7,pasangan combiner,74009,74009,True
8,baris ekspor kategori,73601,73601,True


## P. Pertanyaan evaluasi

1. Map membuat pasangan kata dan angka. Shuffle mengumpulkan kata yang sama. Reduce menjumlahkan angkanya. Pada banyak komputer, data bisa berpindah lewat jaringan saat shuffle.
2. Combiner menjumlahkan dulu di tiap mapper. Untuk penjumlahan, urutan penggabungan tidak mengubah hasil. Rata-rata perlu perlakuan berbeda: simpan jumlah nilai dan banyak datanya, lalu hitung rata-rata di akhir.
3. Input split adalah satu bagian data yang dibaca mapper. Di sini, satu file Parquet menjadi satu split.
4. Data skew berarti pekerjaan tidak terbagi rata. Contohnya, kata di muncul sangat sering sehingga reducer tertentu menerima lebih banyak pasangan. Combiner bisa menguranginya. Cara lain adalah membagi sementara kata yang berat ke beberapa kunci, lalu menggabungkan hasilnya lagi.
5. Hadoop Streaming membaca dan menulis teks melalui stdin dan stdout. Karena itu, mapper dan reducer bisa ditulis dengan Python. Ada tambahan waktu untuk memindahkan teks dan membuka proses dibanding kode yang langsung terhubung dengan Java.
6. Source menerima data, channel menampungnya sementara, dan sink menyimpan data. Batas antrean mencegah terlalu banyak data menumpuk di memori.
7. Sqoop import membawa data dari database ke Hadoop. Export membawa hasil dari Hadoop ke database. Beberapa mapper bisa membaca rentang data berbeda agar pekerjaannya terbagi. Di sini, export dicontohkan dengan SQLite.
8. HDFS menyimpan data di beberapa komputer. YARN membagi sumber daya untuk pekerjaan di cluster. Keduanya tidak dipakai pada percobaan Python dan Hadoop local mode di notebook ini.

## Arsip Hasil

Arsip berisi contoh data mentah, Parquet hasil pembersihan, dan hasil pengolahan.

In [29]:
import zipfile
ZIP_HASIL = Path('BD_A_P05_2411531001_Habibi_Putra_Rizqullah_hasil.zip')
with zipfile.ZipFile(ZIP_HASIL,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in [TERKURASI,SIAP]:
        for p in sorted(folder.rglob('*')):
            if p.is_file(): z.write(p,p.as_posix())
    # Contoh data mentah
    contoh_mentah = mentah.head(100).to_json(orient='records',lines=True,force_ascii=False)
    z.writestr('lapisan_mentah/contoh_100_baris.jsonl',contoh_mentah)
    z.writestr('README.txt','Dataset: '+URL_DATA+'\nHanya untuk pembelajaran/pengumpulan terbatas.\nLapisan mentah hanya contoh 100 baris.\n')
print('Paket:',ZIP_HASIL.resolve(),'| MB:',round(ZIP_HASIL.stat().st_size/1e6,2))



Paket: /content/BD_A_P05_2411531001_Habibi_Putra_Rizqullah_hasil.zip | MB: 59.7


## Referensi
- Modul Praktikum Big Data 05: Pipeline Data Berita, Universitas Andalas.
- Dataset: https://github.com/ibamibrahim/dataset-judul-berita-indonesia
- Hadoop Streaming 3.3.6: https://hadoop.apache.org/docs/r3.3.6/hadoop-streaming/HadoopStreaming.html
- Protokol sitemap: https://www.sitemaps.org/protocol.html

## Menyimpan hasil praktikum

Folder data dan hasil pengolahan disimpan di `MyDrive/BigData/Praktikum5`.

In [30]:
from pathlib import Path
import shutil
import json
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')
DIR_SIMPAN = Path('/content/drive/MyDrive/BigData/Praktikum5')
DIR_SIMPAN.mkdir(parents=True, exist_ok=True)

folder_wajib = [Path('lapisan_mentah'), Path('lapisan_terkurasi'), Path('siap_pakai')]
belum_ada = [str(p) for p in folder_wajib if not p.exists() or not any(p.rglob('*'))]
if belum_ada:
    raise FileNotFoundError('Hasil belum lengkap pada runtime ini: ' + ', '.join(belum_ada))

tersalin = []
for folder in folder_wajib + [Path('contoh_crawl')]:
    if folder.exists():
        tujuan = DIR_SIMPAN / folder.name
        shutil.copytree(folder, tujuan, dirs_exist_ok=True)
        for sumber in folder.rglob('*'):
            if sumber.is_file():
                salinan = tujuan / sumber.relative_to(folder)
                assert salinan.stat().st_size == sumber.stat().st_size
                tersalin.append(str(salinan.relative_to(DIR_SIMPAN)))

nama_zip = 'BD_A_P05_2411531001_Habibi_Putra_Rizqullah_hasil.zip'
for nama in [nama_zip, 'masukan.tsv', 'mapper.py', 'reducer.py', 'hasil_pipe.txt']:
    sumber = Path(nama)
    if sumber.exists():
        tujuan = DIR_SIMPAN / nama
        shutil.copy2(sumber, tujuan)
        assert sumber.stat().st_size == tujuan.stat().st_size
        tersalin.append(nama)

(DIR_SIMPAN / 'daftar_hasil.json').write_text(
    json.dumps({'waktu_simpan': datetime.now().isoformat(), 'jumlah_file': len(tersalin),
                'file': tersalin}, indent=2), encoding='utf-8')
print('Lokasi:', DIR_SIMPAN)
print('Jumlah file yang disalin:', len(tersalin))
for nama in tersalin:
    print('-', nama)
print('Penyimpanan selesai; ukuran file sumber dan salinan sudah sama.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Lokasi: /content/drive/MyDrive/BigData/Praktikum5
Jumlah file yang disalin: 48
- lapisan_mentah/batch_0014.jsonl
- lapisan_mentah/batch_0019.jsonl
- lapisan_mentah/batch_0006.jsonl
- lapisan_mentah/batch_0004.jsonl
- lapisan_mentah/batch_0016.jsonl
- lapisan_mentah/batch_0013.jsonl
- lapisan_mentah/batch_0002.jsonl
- lapisan_mentah/batch_0018.jsonl
- lapisan_mentah/batch_0009.jsonl
- lapisan_mentah/batch_0005.jsonl
- lapisan_mentah/batch_0010.jsonl
- lapisan_mentah/batch_0008.jsonl
- lapisan_mentah/batch_0017.jsonl
- lapisan_mentah/batch_0001.jsonl
- lapisan_mentah/batch_0011.jsonl
- lapisan_mentah/batch_0003.jsonl
- lapisan_mentah/batch_0007.jsonl
- lapisan_mentah/batch_0015.jsonl
- lapisan_mentah/batch_0012.jsonl
- lapisan_terkurasi/kategori=oto/7a54c0c9910d4a0c89d3e626bb5ed4ae-0.parquet
- lapisan_terkurasi/kategori=sport/7a54c0c9910d4a0c89d3e626bb5ed4ae-0.